# Efficient ResNet on CIFAR-10: Control Panel

Everything is driven from here: environment and data checks, model size, training, results, and `submission.csv`.
The code itself lives in `train.py`, `predict.py` and `src/`.

### Two ways to run this notebook
- **GPU Jupyter session** (`c12m85-a100-1` or `g2-standard-12`, 1 GPU): `train()` runs training on this session's GPU in the background. The run survives browser disconnects and kernel restarts, but it stops when the session ends, so request enough hours. Start training soon after the session opens, because GCP shuts down instances whose GPU sits idle for 20 minutes. Delete the session in OOD when you are done, since it uses GPU hours for as long as it is open.
- **CPU Jupyter session** (`interactive` or `n2c48m24`): `train()` queues a Slurm GPU job (`slurm/train.sbatch`) instead. The job keeps running after the session closes.

### One-time setup on HPC
1. **Data:** create `/scratch/$USER/cifar_data/` in OOD Files and upload the competition zip there. It is extracted automatically in step 0.
2. **Code:** in an OOD terminal, run `cd ~/Project && git clone <repo-url>`. Afterwards, `git pull` (cell below) brings in updates.
3. **Python kernel:** use the default `Python 3 (ipykernel)` kernel. Its Python (`/share/apps/pyenv/py3.9`) already has a CUDA build of torch (2.6.0+cu126).

## 0. Session setup and checks

In [ ]:
import csv, json, shutil, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "train.py").is_file(), "Open main.ipynb from the repository root."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ON_HPC = shutil.which("sbatch") is not None
PY = f'"{sys.executable}"'

def sh(cmd):
    """Run a shell command from the repo root and stream its output into the notebook."""
    with subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")
    return p.returncode

def read_log(run_dir):
    with open(run_dir / "log.csv", newline="") as f:
        return list(csv.DictReader(f))

def list_runs(include_smoke=False):
    """Run folders that have a log.csv, sorted by name."""
    return sorted(d for d in (ROOT / "runs").iterdir()
                  if (d / "log.csv").is_file() and (include_smoke or "_smoke_" not in d.name))

print("ON_HPC:", ON_HPC, "| python:", sys.executable)

In [ ]:
# Bring in code changes pushed from the local machine.
sh("git pull --ff-only")

In [ ]:
import torch
from src.utils import env_info, setup_device

device, amp_dtype = setup_device()
info = env_info(device, amp_dtype)
print(json.dumps(info, indent=2))
if ON_HPC and info["torch_cuda_build"] is None:
    print("WARNING: this kernel's torch is a CPU build, so GPU jobs started from it would not use the GPU.")

In [ ]:
from src.data import CACHE_DIR, build_cache, find_data_dir

DATA_DIR = find_data_dir()   # HPC: /scratch/$USER/cifar_data (extracts the zip once); local: ./data
build_cache(DATA_DIR)        # PNG -> cache/*.npy, first time only
print("Data:", DATA_DIR)
print((CACHE_DIR / "meta.json").read_text())

## 1. Model size
Check a configuration against the 5M trainable-parameter limit before spending GPU time on it.

In [ ]:
from src.config import load_config
from src.model import describe

describe(load_config("configs/baseline.json"))
# describe(load_config("configs/baseline.json", ["widths=[40,80,160,320]"]))

## 2. Smoke test (optional, seconds on GPU)
Runs the whole pipeline on 2,000 images for 2 epochs. Use it after changing code, before a real run.

In [ ]:
sh(f"{PY} train.py --config configs/baseline.json --smoke")

## 3. Train
In a GPU session, `train()` starts training here in the background. In a CPU session, it queues a Slurm GPU job.
The first cell defines the helpers. The second starts every experiment in `EXPERIMENTS` that has no run folder yet, so re-running the notebook never starts the same experiment twice.
Each experiment changes one thing relative to the baseline and needs a unique `name`. Several runs can share the A100: each uses about 1 GB of its 40 GB.

In [ ]:
import re
from datetime import datetime

def _train_args(config, overrides, resume):
    return ["--resume", str(resume)] if resume else ["--config", config] + (["--set", *overrides] if overrides else [])

def run_name(config="configs/baseline.json", overrides=(), resume=None):
    """The experiment name a run will get: name= override, else the resumed folder or the config's name."""
    for o in overrides:
        if o.startswith("name="):
            return o.split("=", 1)[1]
    return Path(resume).parent.name if resume else json.loads(Path(config).read_text())["name"]

def train_here(config="configs/baseline.json", overrides=(), resume=None):
    """Train on this session's GPU in the background and return the log path."""
    log = ROOT / "runs" / f"session-{run_name(config, overrides, resume)}-{datetime.now():%m%d-%H%M%S}.out"
    with open(log, "w") as f:
        # A new session keeps training alive through kernel restarts; it ends with the Jupyter session.
        subprocess.Popen([sys.executable, "-u", "train.py", *_train_args(config, overrides, resume)],
                         cwd=ROOT, stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
    print("Training started. Log:", log.relative_to(ROOT))
    return log

def submit(config="configs/baseline.json", overrides=(), resume=None):
    """Queue a Slurm GPU training job and return its job id."""
    if not ON_HPC:
        print("sbatch is not available here; submit from the HPC notebook.")
        return None
    cmd = ["sbatch", "--parsable", f"--export=ALL,PY={sys.executable}", "slurm/train.sbatch",
           *_train_args(config, overrides, resume)]
    out = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(out.stdout + out.stderr, end="")
    return out.stdout.strip() or None

def train(config="configs/baseline.json", overrides=(), resume=None):
    """GPU session: train here. CPU session: queue a Slurm job."""
    run = train_here if torch.cuda.is_available() else submit
    return run(config, overrides, resume)

def has_run(name):
    """True if a run folder <name>_<time> exists (a queued Slurm job has none until it starts)."""
    pattern = re.compile(rf"{re.escape(name)}_\d{{4}}-\d{{6}}")
    return any(pattern.fullmatch(d.name) for d in (ROOT / "runs").iterdir())

In [ ]:
# Overrides relative to configs/baseline.json, one list per experiment, each with a unique name.
EXPERIMENTS = [
    ["cutout=16", "name=cutout16"],
    ["label_smoothing=0.1", "name=ls01"],
    ["widths=[40,80,160,320]", "name=wide40"],
    ["epochs=200", "name=ep200"],
    ["color_jitter=0.3", "name=cj03"],
    ["trivial_augment=true", "name=ta"],
    ["trivial_augment=true", "epochs=200", "name=ta_ep200"],
    ["ema_decay=0.999", "name=ema999"],
]

for overrides in EXPERIMENTS:
    if has_run(run_name(overrides=overrides)):
        print(f"skip {run_name(overrides=overrides)}: already started")
    else:
        train(overrides=overrides)

# To continue an interrupted run: train(resume="runs/<run>/last.pt")

## 4. Monitor
Re-run the cell to refresh. To cancel a Slurm job, run `sh("scancel <jobid>")`. To stop training in a GPU session, run `sh("pkill -f train.py")`.

In [ ]:
import time

def status():
    """GPU load, Slurm queue, progress and estimated finish time of unfinished runs, and recent crash logs."""
    if torch.cuda.is_available():
        sh("nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv")
    if ON_HPC:
        sh("squeue -u $USER")
    for d in list_runs():
        if (d / "summary.json").is_file():
            continue
        log = read_log(d)
        epochs = json.loads((d / "config.json").read_text())["epochs"]
        idle_min = (time.time() - (d / "log.csv").stat().st_mtime) / 60
        val = f"val {float(log[-1]['val_acc']):.2%}" if log and log[-1]["val_acc"] else ""
        if idle_min < 3:
            # Recent epochs give the current speed, which changes as other runs on the GPU finish.
            recent = [float(r["epoch_sec"]) for r in log[-3:]]
            left_min = sum(recent) / len(recent) * (epochs - len(log)) / 60
            state = f"running, ~{left_min:.0f} min left (ends ~{time.strftime('%H:%M', time.localtime(time.time() + left_min * 60))})"
        else:
            state = f"stopped ({idle_min:.0f} min ago)"
        print(f"{d.name:<34} {len(log):>3}/{epochs:<4} {val:<12} {state}")
    # A run that crashes before its first epoch has no folder yet; its error is only in the log file.
    for log in [*(ROOT / "runs").glob("session-*.out"), *(ROOT / "runs").glob("slurm-*.out")]:
        text = log.read_text(errors="replace")
        if "Traceback" in text and time.time() - log.stat().st_mtime < 3600:
            print(f"--- error in {log.name}:\n" + "".join(text.splitlines(keepends=True)[-8:]))

status()

## 5. Results
Copy the finished rows into `experiments.md`.

In [ ]:
def runs_table(include_smoke=False):
    print(f"{'run':<34} {'params':>10} {'epochs':>8} {'best val':>9} {'@ep':>4} {'train':>7} {'min':>6}  status")
    for d in list_runs(include_smoke):
        cfg = json.loads((d / "config.json").read_text())
        env = json.loads((d / "env.json").read_text())
        log = read_log(d)
        vals = [(float(r["val_acc"]), int(r["epoch"])) for r in log if r["val_acc"]]
        best, best_ep = max(vals) if vals else (float("nan"), 0)
        minutes = sum(float(r["epoch_sec"]) for r in log) / 60
        done = "done" if (d / "summary.json").is_file() else "running/stopped"
        print(f"{d.name:<34} {env['n_params']:>10,} {len(log):>3}/{cfg['epochs']:<4} {best:>9.2%} {best_ep:>4} "
              f"{float(log[-1]['train_acc']) if log else 0:>7.2%} {minutes:>6.1f}  {done}")

runs_table()

In [ ]:
import matplotlib.pyplot as plt

def plot_runs(names=None, include_smoke=False):
    """Accuracy and loss curves; names are substrings of run folder names (default: all runs)."""
    runs = [d for d in list_runs(include_smoke) if not names or any(n in d.name for n in names)]
    fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))
    for d in runs:
        log = read_log(d)
        if not log:
            continue
        ep = [int(r["epoch"]) for r in log]
        line, = ax_acc.plot(ep, [float(r["train_acc"]) for r in log], "--", alpha=0.6)
        if log[0]["val_acc"]:
            ax_acc.plot(ep, [float(r["val_acc"]) for r in log], color=line.get_color(), label=d.name)
            ax_loss.plot(ep, [float(r["val_loss"]) for r in log], color=line.get_color(), label=d.name)
        ax_loss.plot(ep, [float(r["train_loss"]) for r in log], "--", color=line.get_color(), alpha=0.6)
    ax_acc.set(title="accuracy (solid = val, dashed = train)", xlabel="epoch")
    ax_loss.set(title="loss (solid = val, dashed = train)", xlabel="epoch")
    ax_acc.legend(fontsize=8)
    plt.tight_layout(); plt.show()

plot_runs()                      # or plot_runs(["baseline_1104", "cutout16"])

## 6. Submission
Picks the finished run with the best validation accuracy, writes `submission.csv` into its folder, and checks the format against `sample_submission.csv`.
Download it from OOD Files and upload it to Kaggle (max 5 submissions per day per team).

In [ ]:
def best_run():
    """Finished run with the highest validation accuracy (full-train runs have none and are skipped)."""
    scored = []
    for d in list_runs():
        if (d / "summary.json").is_file():
            acc = json.loads((d / "summary.json").read_text())["best_val_acc"]
            if acc is not None:
                scored.append((acc, d))
    return max(scored)[1] if scored else None

RUN = best_run()      # or pick one, e.g. ROOT / "runs/baseline_1104-030203"
if RUN is None:
    print("No finished run yet.")
else:
    print("Run:", RUN.name)
    ckpt = RUN / "best.pt" if (RUN / "best.pt").is_file() else RUN / "final.pt"
    sh(f'{PY} predict.py --ckpt "{ckpt}" --tta')